# Odor Intensity and Pleasantness
## https://www.science.org/doi/10.1126/science.aal2014

In [1]:
import math
import random
import bisect
import numpy as np
import pandas as pd

SEED      = 42
random.seed(SEED)
np.random.seed(SEED)

IN_PATH   = "Data/percept_single_with_pubchem.csv"
OUT_INT   = "odor_intensity_175.csv"
OUT_PLEA  = "odor_pleasantness_175.csv"
N_ROWS    = 175
MIN_GAP   = 10.0  

REQ = ["PUBCHEM_NAME", "PUBCHEM_ISOMERIC_SMILES"]

COL_ORDER = [
    "question_ID",
    "compound.name_1",
    "SMILES_1",
    "compound.name_2",
    "SMILES_2",
    "OPTIONS",
    "question_category",
    "prompt.1",
    "prompt.2",
    "answer",
    "other_info",
]

def clean_nonempty(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    ss = str(s).strip()
    return "" if ss.lower() in {"", "nan", "none", "null", "na", "n/a", "nil"} else ss

def rfloat(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def prepare_df(df, metric):
    keep = [c for c in (REQ + [metric]) if c in df.columns]
    miss = [c for c in (REQ + [metric]) if c not in df.columns]
    if miss:
        raise SystemExit(f"Missing required column(s) for {metric}: {miss}")
    d = df[keep].copy()
    d["PUBCHEM_NAME"] = d["PUBCHEM_NAME"].apply(clean_nonempty)
    d["PUBCHEM_ISOMERIC_SMILES"] = d["PUBCHEM_ISOMERIC_SMILES"].apply(clean_nonempty)
    d[metric] = d[metric].apply(rfloat)
    d = d[(d["PUBCHEM_NAME"] != "") & (d["PUBCHEM_ISOMERIC_SMILES"] != "")]
    d = d.dropna(subset=[metric]).reset_index(drop=True)
    return d

def split_halves(df, metric):
    med = df[metric].median()
    bottom = df[df[metric] < med].copy()
    top    = df[df[metric] >= med].copy()
    return bottom, top, med

def all_valid_pairs(bottom, top, metric, min_gap=10.0):

    if bottom.empty or top.empty:
        return [], bottom, top

    b = bottom.sort_values(by=metric).reset_index(drop=True)
    t = top.sort_values(by=metric).reset_index(drop=True)

    t_vals = t[metric].to_numpy()
    pairs = []

    for i in range(len(b)):
        need = float(b.loc[i, metric]) + min_gap
        j = bisect.bisect_left(t_vals, need)
        for k in range(j, len(t)):
            pairs.append((i, k))

    return pairs, b, t

def build_rows_from_pairs(pairs, b, t, metric, n_rows):

    if not pairs:
        return []

    if len(pairs) >= n_rows:
        chosen = random.sample(range(len(pairs)), n_rows)
    else:
        chosen = list(range(len(pairs)))
        while len(chosen) < n_rows:
            chosen.append(random.randrange(len(pairs)))

    rows = []
    for idx in chosen:
        i, k = pairs[idx]
        a = b.loc[i]
        c = t.loc[k]

        if random.random() < 0.5:
            r1, r2 = a, c
        else:
            r1, r2 = c, a

        name1 = str(r1["PUBCHEM_NAME"])
        name2 = str(r2["PUBCHEM_NAME"])
        smi1  = str(r1["PUBCHEM_ISOMERIC_SMILES"])
        smi2  = str(r2["PUBCHEM_ISOMERIC_SMILES"])
        v1    = float(r1[metric])
        v2    = float(r2[metric])

        if v1 > v2:
            win_smiles, win_name = smi1, name1
        elif v2 > v1:
            win_smiles, win_name = smi2, name2
        else:
            win_smiles, win_name = smi1, name1

        rows.append({
            "compound.name_1": name1,
            "SMILES_1": smi1,
            "compound.name_2": name2,
            "SMILES_2": smi2,
            "OPTIONS": f"{{{smi1};{name1}}};{{{smi2};{name2}}}",
            "answer": f"{win_smiles};{win_name}",
            "other_info": f"SMILES_1 {metric}={v1};SMILES_2 {metric}={v2}",
        })

    random.shuffle(rows)
    return rows

def finalize_prompts(rows, metric):
    if metric == "INTENSITY":
        category = "odor_intensity"
        p1_tail = ("Select only one of the SMILES. If you had to rate the intensity of these molecules from 0 (extremely low) to 100 (highly intense), "
                   "what would you assign to each? Respond with the selected compound name (the one with higher intensity), followed by two intensity values "
                   "in the order the molecules are listed. Use semicolons (;) as separators. Do not write any comments.")
        p2_tail = ("Select only one of the compound names. If you had to rate the intensity of these molecules from 0 (extremely low) to 100 (highly intense), "
                   "what would you assign to each? Respond with the selected compound name (the one with higher intensity), followed by two intensity values "
                   "in the order the molecules are listed. Use semicolons (;) as separators. Do not write any comments.")
    else:
        category = "odor_pleasantness"
        p1_tail = ("Select only one of the SMILES. If you had to rate the pleasantness of these molecules from 0 (extremely unpleasant) to 100 (highly pleasant), "
                   "what would you assign to each? Respond with the selected compound name (the one with higher pleasantness), followed by two pleasantness values "
                   "in the order the molecules are listed. Use semicolons (;) as separators. Do not write any comments.")
        p2_tail = ("Select only one of the compound names. If you had to rate the pleasantness of these molecules from 0 (extremely unpleasant) to 100 (highly pleasant), "
                   "what would you assign to each? Respond with the selected compound name (the one with higher pleasantness), followed by two pleasantness values "
                   "in the order the molecules are listed. Use semicolons (;) as separators. Do not write any comments.")

    for r in rows:
        r["question_category"] = category
        r["prompt.1"] = (f"Which molecule is likely to smell more {'intense' if metric=='INTENSITY' else 'pleasant'} "
                         f"to humans: {r['SMILES_1']} or {r['SMILES_2']}? " + p1_tail)
        r["prompt.2"] = (f"Which molecule is likely to smell more {'intense' if metric=='INTENSITY' else 'pleasant'} "
                         f"to humans: {r['compound.name_1']} or {r['compound.name_2']}? " + p2_tail)

def build_and_write(df, metric, out_path):
    d = prepare_df(df, metric)
    bottom, top, med = split_halves(d, metric)
    print(f"[{metric}] median={med:.4g}  bottom={len(bottom)}  top={len(top)}  (MIN_GAP={MIN_GAP})")

    pairs, b_sorted, t_sorted = all_valid_pairs(bottom, top, metric, min_gap=MIN_GAP)
    print(f"[{metric}] valid bottom↔top pairs with Δ≥{MIN_GAP}: {len(pairs)}")

    if not pairs:
        raise SystemExit(f" No valid pairs for {metric} with Δ≥{MIN_GAP}.")

    rows = build_rows_from_pairs(pairs, b_sorted, t_sorted, metric, N_ROWS)
    if not rows:
        raise SystemExit(f" Could not assemble rows for {metric}.")

    finalize_prompts(rows, metric)

    out_df = pd.DataFrame(rows)
    out_df.insert(0, "question_ID", np.arange(1, len(out_df) + 1, dtype=int))
    out_df = out_df[COL_ORDER]

    out_df.to_csv(out_path, index=False)
    print(f" Wrote {out_path} (n={len(out_df)})")

def main():
    df = pd.read_csv(IN_PATH)
    build_and_write(df, "INTENSITY", OUT_INT)
    build_and_write(df, "PLEASANTNESS", OUT_PLEA)

if __name__ == "__main__":
    main()


[INTENSITY] median=56.68  bottom=169  top=169  (MIN_GAP=10.0)
[INTENSITY] valid bottom↔top pairs with Δ≥10.0: 27295
 Wrote odor_intensity_175.csv (n=175)
[PLEASANTNESS] median=42.71  bottom=169  top=169  (MIN_GAP=10.0)
[PLEASANTNESS] valid bottom↔top pairs with Δ≥10.0: 24728
 Wrote odor_pleasantness_175.csv (n=175)
